# EEG Scalp-to-In-Ear Downsampling — Results Analysis

This notebook loads trained models, evaluates them, and generates all figures.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.dataset import EEGDataset
from src.models import LinearSpatialFilter, SpatioTemporalFIR, ConvEncoder, ClosedFormLinear
from src.metrics.evaluation import compute_all_metrics, format_metrics_table
from src.visualize import (
    plot_time_traces, plot_psd, plot_band_power_scatter,
    plot_spatial_weights, plot_spatial_topomaps, plot_fir_filters,
    plot_coherence, generate_latex_table
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load Data

In [ ]:
data_path = Path('../data/processed/data.h5')
test_ds = EEGDataset.from_hdf5(data_path, 'test')
train_ds = EEGDataset.from_hdf5(data_path, 'train')

print(f'Test set: {len(test_ds)} windows')
print(f'Scalp shape: {test_ds.scalp.shape}')
print(f'In-ear shape: {test_ds.inear.shape}')

## 2. Load and Evaluate Models

In [ ]:
device = torch.device('cpu')
ckpt_dir = Path('../results/checkpoints')
fig_dir = Path('../results/figures')
fig_dir.mkdir(parents=True, exist_ok=True)

# Collect predictions from all models
model_preds = {}
model_metrics = {}

target = test_ds.inear.numpy()
scalp = test_ds.scalp

In [ ]:
# Closed-form baseline
cf_model = ClosedFormLinear()
cf_model.fit(train_ds.scalp.numpy(), train_ds.inear.numpy())
with torch.no_grad():
    pred_cf = cf_model(scalp).numpy()
model_preds['Closed-form'] = pred_cf
model_metrics['Closed-form'] = compute_all_metrics(pred_cf, target)
print(format_metrics_table(model_metrics['Closed-form']))

In [ ]:
# Model 1: Linear Spatial Filter
if (ckpt_dir / 'linear_spatial_best.pt').exists():
    m1 = LinearSpatialFilter()
    m1.load_state_dict(torch.load(ckpt_dir / 'linear_spatial_best.pt', weights_only=True))
    m1.eval()
    with torch.no_grad():
        pred_m1 = m1(scalp).numpy()
    model_preds['Linear Spatial'] = pred_m1
    model_metrics['Linear Spatial'] = compute_all_metrics(pred_m1, target)
    print(format_metrics_table(model_metrics['Linear Spatial']))
else:
    print('No checkpoint found for Linear Spatial Filter')

In [ ]:
# Model 2: FIR Filter
if (ckpt_dir / 'fir_filter_best.pt').exists():
    m2 = SpatioTemporalFIR(filter_length=65, mode='acausal')
    m2.load_state_dict(torch.load(ckpt_dir / 'fir_filter_best.pt', weights_only=True))
    m2.eval()
    with torch.no_grad():
        pred_m2 = m2(scalp).numpy()
    model_preds['FIR Filter'] = pred_m2
    model_metrics['FIR Filter'] = compute_all_metrics(pred_m2, target)
    print(format_metrics_table(model_metrics['FIR Filter']))
else:
    print('No checkpoint found for FIR Filter')

In [ ]:
# Model 3: Conv Encoder
if (ckpt_dir / 'conv_encoder_best.pt').exists():
    m3 = ConvEncoder(H=64, K=17, N_blocks=4, dropout=0.1)
    m3.load_state_dict(torch.load(ckpt_dir / 'conv_encoder_best.pt', weights_only=True))
    m3.eval()
    with torch.no_grad():
        pred_m3 = m3(scalp).numpy()
    model_preds['Conv Encoder'] = pred_m3
    model_metrics['Conv Encoder'] = compute_all_metrics(pred_m3, target)
    print(format_metrics_table(model_metrics['Conv Encoder']))
else:
    print('No checkpoint found for Conv Encoder')

## 3. Visualizations

In [ ]:
# Time traces (using best available model)
best_model_name = max(model_metrics, key=lambda k: model_metrics[k]['pearson_r_mean'])
best_pred = model_preds[best_model_name]
print(f'Best model: {best_model_name}')

plot_time_traces(best_pred, target, save_path=fig_dir / 'time_traces.png')
plot_psd(best_pred, target, save_path=fig_dir / 'psd.png')
plot_band_power_scatter(best_pred, target, save_path=fig_dir / 'band_power_scatter.png')
plot_coherence(best_pred, target, save_path=fig_dir / 'coherence.png')

In [ ]:
# Spatial weights (closed-form)
plot_spatial_weights(cf_model.weight_matrix.numpy(), save_path=fig_dir / 'spatial_weights.png')
plot_spatial_topomaps(cf_model.weight_matrix.numpy(), save_path=fig_dir / 'spatial_topomaps.png')

In [ ]:
# FIR filters
if 'FIR Filter' in model_preds:
    plot_fir_filters(m2.filters.numpy(), save_path=fig_dir / 'fir_filters.png')

In [ ]:
# LaTeX comparison table
latex = generate_latex_table(model_metrics, save_path=fig_dir / 'comparison_table.tex')
print(latex)

## 4. Summary

In [ ]:
print('=== Model Comparison ===')
for name, m in model_metrics.items():
    print(f"{name:20s}  r={m['pearson_r_mean']:.4f}  SNR={m['snr_db_mean']:.2f} dB  RMSE={m['rmse_mean']:.4f}")